# EDA — feature engineering + preprocessing

Same log as my `eda.ipynb` (Project 1): we **drop** / **keep** column by column, then preprocess.
At the end: `dropped_columns.csv` and `kept_columns.csv`.


In [ ]:
import csv
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

# notebook lives in notebooks/feature_selection/
NB_DIR = Path.cwd()
if NB_DIR.name == "feature_selection":
    PROJ = NB_DIR.parent.parent  # repo root
    OUT = NB_DIR
elif NB_DIR.name == "notebooks":
    PROJ = NB_DIR.parent
    OUT = NB_DIR / "feature_selection"
else:
    PROJ = Path.cwd()
    OUT = PROJ / "notebooks" / "feature_selection"
sys.path.insert(0, str(PROJ))

CANDIDATES = [
    PROJ / "data" / "dataset_unzipped" / "dataset",
    PROJ / "data" / "dataset",
    PROJ / "data",
]
DATA = next((p for p in CANDIDATES if (p / "x_train.csv").exists()), None)
assert DATA is not None, "put the CSVs in data/dataset_unzipped/dataset/"

X_TRAIN = DATA / "x_train.csv"
Y_TRAIN = DATA / "y_train.csv"
X_TEST = DATA / "x_test.csv"
OUT.mkdir(parents=True, exist_ok=True)

print("DATA :", DATA.resolve())
print("OUT  :", OUT.resolve())


## 1. Data size

I count the rows and columns, and check that train / test / labels match.

In [ ]:
def header(path):
    with path.open(newline="") as f:
        return next(csv.reader(f))


def n_rows(path):
    with path.open(newline="") as f:
        return sum(1 for _ in f) - 1


x_cols = header(X_TRAIN)
print("X cols", len(x_cols), " | y cols", header(Y_TRAIN))
print("train/test same cols?", x_cols == header(X_TEST))
print("n train", n_rows(X_TRAIN), " n y", n_rows(Y_TRAIN), " n test", n_rows(X_TEST))

## 2. One person

I look at the first row to see what an observation looks like (and the empty cells).

In [ ]:
with X_TRAIN.open(newline="") as f:
    r = csv.reader(f)
    names = next(r)
    row = next(r)

with Y_TRAIN.open(newline="") as f:
    r = csv.reader(f)
    next(r)
    y0 = next(r)

print("y", y0)
n_empty = sum(v == "" for v in row)
print("empty on this row", n_empty, "/", len(row))
for name, v in zip(names, row):
    print(f"{name:12} {v if v else '∅'}")

## 3. Label imbalance

I count `-1` (no MICHD) and `+1` (MICHD). If there are many more `-1`, a model that always says "no" already gets good accuracy: it is not a good criterion.

In [ ]:
y = []
with Y_TRAIN.open(newline="") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        y.append(float(row[1]))
y = np.array(y)

labels, counts = np.unique(y, return_counts=True)
print(f"{'label':>8}  {'n':>10}  {'%':>7}")
for lab, n in zip(labels, counts):
    print(f"{lab:8g}  {n:10,}  {100 * n / len(y):6.2f}%")
print(f"\nratio -1 / +1 = {counts[labels == -1][0] / counts[labels == 1][0]:.2f}")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["-1 (no)", "+1 (yes)"], [counts[labels == -1][0], counts[labels == 1][0]], color=["#7f8c8d", "#c0392b"])
ax.set(ylabel="number of people", title="_MICHD")
plt.tight_layout()
plt.show()

## 4. Remove what is not a risk factor

I only keep what describes the person (health, habits, age…). I remove IDs, dates, phone, survey weights, screenings, etc.

In [ ]:
feature_names = np.array(header(X_TRAIN))

# not a risk factor for MICHD (survey / identifier / other topic)
NOT_RISK = {
    # identifiers + interview date / version
    "Id", "SEQNO", "_PSU",
    "FMONTH", "IDATE", "IMONTH", "IDAY", "IYEAR",
    "DISPCODE", "QSTVER", "QSTLANG", "MSCODE",
    # how the person was reached (landline / cell phone)
    "CTELENUM", "PVTRESD1", "COLGHOUS", "STATERES", "CELLFON3", "LADULT",
    "NUMADULT", "NUMMEN", "NUMWOMEN", "NUMHHOL2", "NUMPHON2", "CPDEMO1",
    "CTELNUM1", "CELLFON2", "CADULT", "PVTRESD2", "CCLGHOUS", "CSTATE",
    "LANDLINE", "HHADULT",
    # CDC survey weights and sampling design
    "_STSTR", "_STRWT", "_RAWRAKE", "_WT2RAKE",
    "_CHISPNC", "_CRACE1", "_CPRACE", "_CLLCPWT",
    "_DUALUSE", "_DUALCOR", "_LLCPWT",
    # questions about a child in the household (not about the adult)
    "RCSGENDR", "RCSRLTN2", "CASTHDX2", "CASTHNO2",
    # technical flags (response quality, not a risk)
    "_MISFRTN", "_MISVEGN", "_FRTRESP", "_VEGRESP",
    "_FRT16", "_VEG23", "_FRUITEX", "_VEGETEX", "PAMISS1_", "FC60_",
    # road safety / screenings / tests: different domain logic
    "SEATBELT", "_RFSEAT2", "_RFSEAT3",
    "FLUSHOT6", "FLSHTMY2", "IMFVPLAC", "PNEUVAC3", "_FLSHOT6", "_PNEUMO2",
    "HIVTST6", "HIVTSTD3", "WHRTST10", "_AIDTST3",
    "TETANUS", "HPVADVC2", "HPVADSHT", "SHINGLE2",
    "HADMAM", "HOWLONG", "HADPAP2", "LASTPAP2", "HPVTEST", "HPLSTTST",
    "HADHYST2", "PROFEXAM", "LENGEXAM",
    "BLDSTOOL", "LSTBLDS3", "HADSIGM3", "HADSGCO1", "LASTSIG3",
    "PCPSAAD2", "PCPSADI1", "PCPSARE1", "PSATEST1", "PSATIME",
    "PCPSARS1", "PCPSADE1", "PCDMDECN",
}

unknown = NOT_RISK - set(feature_names)
drop = [n for n in feature_names if n in NOT_RISK]
keep = [n for n in feature_names if n not in NOT_RISK]

print("unknown names (should be empty):", unknown)
print(f"drop {len(drop)}  |  keep {len(keep)}  |  total {len(feature_names)}")
print("\n--- drop ---")
print(", ".join(drop))
print("\n--- keep (factors / health / lifestyle / demo) ---")
print(", ".join(keep))

## 5. Three kinds of missing values

Empty = question not asked. 7/9 (or 77/99) = don't know / refused. `88` = no days, it is not a missing value (we will recode it as 0). We drop a column only if empty + refused exceeds 90 %.

In [ ]:
# Count the codes, then decide the type (7 is not always a refusal: EMPLOY1=7 → retired, age=77 → 77 years old)
NONE_88 = {"PHYSHLTH", "MENTHLTH", "POORHLTH", "CHILDREN"}
NONE_888 = {"ALCDAY5"}
CODES = (7, 9, 77, 88, 99, 777, 888, 999, 900, 99900)

keep_names = np.array(keep)
idx = {n: j for j, n in enumerate(feature_names)}
keep_j = np.array([idx[n] for n in keep_names])
n_keep = len(keep_names)

n_empty = np.zeros(n_keep, dtype=np.int64)
code_count = {c: np.zeros(n_keep, dtype=np.int64) for c in CODES}
is_floatish = np.zeros(n_keep, dtype=bool)
vmax = np.zeros(n_keep)
n_rows = 0

with X_TRAIN.open(newline="") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        n_rows += 1
        for i, j in enumerate(keep_j):
            raw = row[j].strip()
            if raw == "":
                n_empty[i] += 1
                continue
            try:
                x = float(raw)
            except ValueError:
                continue
            if not x.is_integer():
                is_floatish[i] = True
            if x in code_count:
                code_count[x][i] += 1
            if x not in (7, 9, 77, 88, 99, 777, 888, 999, 900, 99900) and x > vmax[i]:
                vmax[i] = x

n_none = np.zeros(n_keep, dtype=np.int64)
n_dk = np.zeros(n_keep, dtype=np.int64)
for i, name in enumerate(keep_names):
    if name in NONE_88:
        n_none[i] = code_count[88][i]
        n_dk[i] = code_count[77][i] + code_count[99][i]
    elif name in NONE_888:
        n_none[i] = code_count[888][i]
        n_dk[i] = code_count[777][i] + code_count[999][i]
    elif name == "DROCDY3_":
        n_dk[i] = code_count[900][i]
    elif name == "_DRNKWEK":
        n_dk[i] = code_count[99900][i]
    elif is_floatish[i] or vmax[i] > 30:
        n_dk[i] = 0
    elif vmax[i] <= 5:
        n_dk[i] = code_count[7][i] + code_count[9][i] + code_count[77][i] + code_count[99][i]
    else:
        n_dk[i] = code_count[9][i] + code_count[77][i] + code_count[99][i]

empty_rate = n_empty / n_rows
none_rate = n_none / n_rows
dk_rate = n_dk / n_rows
missing_rate = empty_rate + dk_rate

print(f"{n_rows:,} rows, {n_keep} keep columns\n")
print(f"{'missing':22} {'meaning':32} {'mean'}")
print(f"{'1. empty':22} {'question not asked':32} {100 * empty_rate.mean():5.1f} %")
print(f"{'2. DK / refused':22} {'7/9 or 77/99 depending on col':32} {100 * dk_rate.mean():5.1f} %")
print(f"{'3. 88 / 888':22} {'none → will be recoded to 0':32} {100 * none_rate.mean():5.1f} %")
print(f"{'1 + 2':22} {'true missing (for threshold)':32} {100 * missing_rate.mean():5.1f} %")

Left: one bar per column (grey = empty, orange = refused). Above the 90 % line, we drop. Right: how many columns are almost full vs almost empty.

In [ ]:
MISSING_THRESHOLD = 0.90  # working heuristic, not a theorem

order = np.argsort(-missing_rate)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(np.arange(n_keep), 100 * empty_rate[order], width=1.0, color="#7f8c8d", label="1. empty")
axes[0].bar(np.arange(n_keep), 100 * dk_rate[order], width=1.0, bottom=100 * empty_rate[order], color="#e67e22", label="2. DK / refused")
axes[0].axhline(100 * MISSING_THRESHOLD, color="black", ls="--", lw=1, label=f"threshold {100 * MISSING_THRESHOLD:.0f} %")
axes[0].set(xlabel="keep columns (sorted)", ylabel="%", title="True missing = empty + DK/refused", ylim=(0, 100))
axes[0].legend(fontsize=8)

axes[1].hist(100 * missing_rate, bins=20, color="#2980b9", edgecolor="white")
axes[1].axvline(100 * MISSING_THRESHOLD, color="#c0392b", ls="--", label=f"{100 * MISSING_THRESHOLD:.0f} %")
axes[1].set(xlabel="% missing (1+2)", ylabel="# columns", title="Distribution")
axes[1].legend()
plt.tight_layout()
plt.show()

print("columns with a real 88/888 (none):")
for i in np.where(none_rate > 0)[0]:
    print(f"  {keep_names[i]:12}  {100 * none_rate[i]:5.1f}%  'none'   (do not drop for this)")

List of the columns above the threshold: too few answers to keep them.

In [ ]:
drop_miss = missing_rate >= MISSING_THRESHOLD
keep2 = keep_names[~drop_miss]
drop2 = keep_names[drop_miss][np.argsort(-missing_rate[drop_miss])]

print(f"threshold {100 * MISSING_THRESHOLD:.0f}% on (empty + DK/refused)")
print(f"drop {int(drop_miss.sum())}  |  keep {len(keep2)}\n")
print(f"{'feature':12}  {'empty':>6}  {'DK':>6}  {'none':>6}  {'1+2':>6}")
print("-" * 44)
for name in drop2:
    i = int(np.where(keep_names == name)[0][0])
    print(f"{name:12}  {100 * empty_rate[i]:5.1f}%  {100 * dk_rate[i]:5.1f}%  {100 * none_rate[i]:5.1f}%  {100 * missing_rate[i]:5.1f}%")

## 6. Recode the missing values

Empty and refused → missing (`nan`). `88` / `888` → `0` (none). Without this, the model thinks 88 = many sick days.

In [ ]:
pos = {n: i for i, n in enumerate(keep_names)}


def recode_value(name, raw):
    if raw.strip() == "":
        return np.nan
    try:
        x = float(raw)
    except ValueError:
        return np.nan
    i = pos[name]
    if name in NONE_88 and x == 88:
        return 0.0
    if name in NONE_888 and x == 888:
        return 0.0
    if name == "DROCDY3_" and x == 900:
        return np.nan
    if name == "_DRNKWEK" and x == 99900:
        return np.nan
    if is_floatish[i] or vmax[i] > 30:
        return x
    if vmax[i] <= 5 and x in (7, 9, 77, 99):
        return np.nan
    if vmax[i] > 5 and x in (9, 77, 99):
        return np.nan
    return x


# example on 3 columns, before / after
demo = ["PHYSHLTH", "CHILDREN", "GENHLTH"]
demo = [n for n in demo if n in set(keep2)]
before = {n: [] for n in demo}
after = {n: [] for n in demo}

with X_TRAIN.open(newline="") as f:
    reader = csv.reader(f)
    next(reader)
    for k, row in enumerate(reader):
        if k >= 8000:
            break
        for n in demo:
            raw = row[idx[n]]
            before[n].append(raw.strip())
            after[n].append(recode_value(n, raw))

print("sample of 8000 rows\n")
for n in demo:
    b = before[n]
    a = np.array(after[n], dtype=float)
    print(n)
    print(f"  before  # of '88' = {sum(v == '88' for v in b)}   empty = {sum(v == '' for v in b)}")
    print(f"  after   # of 0    = {int(np.nansum(a == 0))}   nan   = {int(np.isnan(a).sum())}")
    print()

## 7. Nature of the kept columns

I classify what remains: binary, category, ordinal, count or continuous. This tells us whether to one-hot, keep the order, or scale.

In [ ]:
ORDINAL = {
    "GENHLTH", "EDUCA", "INCOME2", "CHECKUP1", "CHILDREN",
    "_AGEG5YR", "_AGE65YR", "_AGE_G", "_BMI5CAT", "_EDUCAG", "_INCOMG",
    "_CHLDCNT", "_SMOKER3", "_PACAT1", "_PAREC1",
}
COUNT = {"PHYSHLTH", "MENTHLTH", "POORHLTH", "CHILDREN", "JOINPAIN"}
# the rest: heuristic on the already recoded values (sample)

n_uniq = {}
n_nan = {}
floatish2 = {}
sample_n = 12000
vals = {n: [] for n in keep2}

with X_TRAIN.open(newline="") as f:
    reader = csv.reader(f)
    next(reader)
    for k, row in enumerate(reader):
        if k >= sample_n:
            break
        for n in keep2:
            vals[n].append(recode_value(n, row[idx[n]]))

kinds = {}
for n in keep2:
    a = np.array(vals[n], dtype=float)
    finite = a[np.isfinite(a)]
    n_nan[n] = int(np.isnan(a).sum())
    n_uniq[n] = len(np.unique(np.round(finite, 6))) if len(finite) else 0
    floatish2[n] = bool(np.any(~np.equal(finite, np.round(finite)))) if len(finite) else False
    if n in ORDINAL:
        kinds[n] = "ordinal"
    elif n in COUNT:
        kinds[n] = "count"
    elif n_uniq[n] <= 2:
        kinds[n] = "binary"
    elif floatish2[n] or n_uniq[n] >= 40:
        kinds[n] = "continuous"
    else:
        kinds[n] = "categorical"

order_k = ["binary", "categorical", "ordinal", "count", "continuous"]
print(f"on {sample_n} rows, {len(keep2)} keep2 columns\n")
print(f"{'type':12}  {'n':>4}")
for k in order_k:
    print(f"{k:12}  {sum(v == k for v in kinds.values()):4d}")

print()
for k in order_k:
    names_k = [n for n in keep2 if kinds[n] == k]
    print(f"=== {k} ({len(names_k)}) ===")
    print("  " + ", ".join(names_k))
    print()

In [ ]:
counts_k = [sum(v == k for v in kinds.values()) for k in order_k]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(order_k, counts_k, color="#2980b9")
ax.set(ylabel="# columns", title="Types after recoding (keep2)")
plt.tight_layout()
plt.show()

## 8. One version per concept

I group the columns that are too strongly related (correlation for numbers, Cramér's V for categories). Within a group I keep one — preferably the CDC `_` version, if it exists.

In [ ]:
ASSOC_THRESHOLD = 0.75
cols = list(keep2)
Xc = np.column_stack([np.array(vals[n], dtype=float) for n in cols])
n_c = len(cols)


def pearson_abs(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 80:
        return np.nan
    if np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return abs(float(np.corrcoef(a[m], b[m])[0, 1]))


def cramers_v(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 80:
        return np.nan
    aa, bb = a[m], b[m]
    ua, ub = np.unique(aa), np.unique(bb)
    if len(ua) < 2 or len(ub) < 2:
        return np.nan
    ia = np.searchsorted(ua, aa)
    ib = np.searchsorted(ub, bb)
    table = np.zeros((len(ua), len(ub)))
    np.add.at(table, (ia, ib), 1)
    s0, s1 = table.sum(1, keepdims=True), table.sum(0, keepdims=True)
    exp = s0 * s1 / table.sum()
    with np.errstate(divide="ignore", invalid="ignore"):
        chi2 = np.nansum((table - exp) ** 2 / exp)
    k = min(len(ua) - 1, len(ub) - 1)
    return float(np.sqrt(chi2 / (table.sum() * k))) if k else np.nan


def to_codes(a, kind, nuniq):
    if kind in ("continuous", "count") and nuniq > 12:
        fin = a[np.isfinite(a)]
        qs = np.quantile(fin, [0.2, 0.4, 0.6, 0.8])
        return np.where(np.isfinite(a), np.digitize(a, qs), np.nan)
    return a


assoc = np.full((n_c, n_c), np.nan)
for i in range(n_c):
    assoc[i, i] = 1.0
    ki, ni = kinds[cols[i]], n_uniq[cols[i]]
    for j in range(i + 1, n_c):
        kj, nj = kinds[cols[j]], n_uniq[cols[j]]
        both_num = ki in ("continuous", "count") and kj in ("continuous", "count")
        if both_num:
            v = pearson_abs(Xc[:, i], Xc[:, j])
        else:
            left = to_codes(Xc[:, i], ki, ni)
            right = to_codes(Xc[:, j], kj, nj)
            v = cramers_v(left, right)
        assoc[i, j] = assoc[j, i] = v

print(f"association threshold = {ASSOC_THRESHOLD}  |  {n_c} columns, sample {len(vals[cols[0]])}")

In [ ]:
parent = np.arange(n_c)


def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a


for i in range(n_c):
    for j in range(i + 1, n_c):
        if np.isfinite(assoc[i, j]) and assoc[i, j] >= ASSOC_THRESHOLD:
            parent[find(j)] = find(i)

groups, seen = [], {}
for i in range(n_c):
    r = find(i)
    seen.setdefault(r, []).append(i)
groups = [g for g in seen.values() if len(g) >= 2]
groups.sort(key=len, reverse=True)

print(f"{len(groups)} redundant groups, {sum(len(g) for g in groups)} columns in them")
print(f"{n_c - sum(len(g) for g in groups)} standalone columns (kept)\n")
for g in groups:
    print("—", ", ".join(cols[i] for i in g))

In [ ]:
# domain priority: the already-clean CDC version, otherwise fewest nan
PREF = [
    "_SMOKER3", "_BMI5", "_PACAT1", "_RFHYPE5", "_RFCHOL", "_CHOLCHK",
    "_DRDXAR1", "_ASTHMS1", "_EDUCAG", "_INCOMG", "_AGE80", "_RACE",
    "_FRUTSUM", "_VEGESUM", "_TOTINDA", "_RFBING5", "_RFHLTH",
    "DIABETE3", "GENHLTH",
]


def pick(g):
    names_g = [cols[i] for i in g]
    for p in PREF:
        if p in names_g:
            return p
    scored = sorted(g, key=lambda i: (n_nan[cols[i]], -n_uniq[cols[i]], cols[i]))
    return cols[scored[0]]


in_group = set()
keep3 = []
print(f"{'I keep':12}  {'I drop (same concept, correlated >= 0.75)'}")
print("-" * 70)
for g in groups:
    chosen = pick(g)
    others = [cols[i] for i in g if cols[i] != chosen]
    keep3.append(chosen)
    in_group.update(cols[i] for i in g)
    print(f"{chosen:12}  {', '.join(others)}")

alone = [n for n in cols if n not in in_group]
keep3.extend(alone)
print(f"\nkeep3 = {len(keep3)} columns  (of which {len(alone)} without duplicates)")
for i in range(0, len(keep3), 6):
    print("  " + ", ".join(keep3[i:i + 6]))


## 9. Domain pass on keep3

The correlation step kept too much. I manually remove what still duplicates `_PACAT1` / `_BMI5` / diet, plus `ALCDAY5` and the skip questions.

In [ ]:
# still the same concept, or skip / module, or awkward raw code
DROP_METIER = {
    # already covered by _BMI5
    "WEIGHT2", "HEIGHT3", "HTIN4", "HTM4", "WTKG3",
    # already covered by _PACAT1
    "ACTIN21_", "EXERHMM1", "EXERHMM2", "EXRACT11", "EXEROFT1",
    "EXRACT21", "EXEROFT2", "METVL11_", "METVL21_", "PAFREQ1_", "PAFREQ2_",
    "PADUR1_", "PADUR2_",
    # already covered by _FRUTSUM / _VEGESUM
    "FRUITJU1", "FRUIT1", "FVBEANS", "FVGREEN", "FVORANG", "VEGETAB1",
    "FTJUDA1_", "FRUTDA1_", "BEANDAY_", "GRENDAY_", "ORNGDAY_", "VEGEDA1_",
    # alcohol: keep _RFBING5 + _RFDRHV5, not the raw code
    "ALCDAY5", "AVEDRNK2",
    # detailed tobacco (we have _SMOKER3)
    "STOPSMK2", "LASTSMK2",
    # skip / subgroup (often empty, diagnosis already covered elsewhere)
    "BPMEDS", "ASTHNOW", "DIABAGE2", "PREGNANT", "POORHLTH",
    "PDIABTST", "PREDIAB1", "JOINPAIN",
    "CAREGIV1", "CRGVEXPT", "CIMEMLOS",
    "_STATE",
}

unknown = DROP_METIER - set(keep3)
keep4 = [n for n in keep3 if n not in DROP_METIER]
drop4 = [n for n in keep3 if n in DROP_METIER]

print("names not in keep3 (ok if already dropped):", unknown)
print(f"keep3 {len(keep3)}  →  drop {len(drop4)}  →  keep4 {len(keep4)}\n")
print("removed:")
print("  " + ", ".join(drop4))
print("\nkept:")
for i in range(0, len(keep4), 6):
    print("  " + ", ".join(keep4[i:i + 6]))

## 10. Association with MICHD

For each keep4 column, I compare the `-1` and the `+1`. The larger the gap, the more useful the feature looks.

In [ ]:
n_s = len(vals[keep4[0]])
y_s = y[:n_s]
base = float((y_s == 1).mean())
scores = []

for n in keep4:
    x = np.array(vals[n], dtype=float)
    m = np.isfinite(x)
    if m.sum() < 80:
        continue
    xx, yy = x[m], y_s[m]
    k = kinds.get(n, "continuous")
    if k in ("binary", "categorical", "ordinal") or n_uniq[n] <= 12:
        levels = np.unique(xx)
        rates = [float((yy[xx == lv] == 1).mean()) for lv in levels]
        score = max(rates) - min(rates)
        how = "gap in % MICHD across levels"
    else:
        if np.std(xx) == 0:
            continue
        score = abs(float(np.corrcoef(xx, (yy == 1).astype(float))[0, 1]))
        how = "|correlation| with MICHD"
    scores.append((score, n, k, how))

scores.sort(reverse=True)
print(f"sample {n_s}  |  % MICHD = {100 * base:.1f}%\n")
print(f"{'feature':12}  {'type':10}  {'score':>6}  how it is measured")
print("-" * 62)
for score, n, k, how in scores:
    print(f"{n:12}  {k:10}  {score:6.3f}  {how}")

In [ ]:
top = scores[:15]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([n for _, n, _, _ in top][::-1], [s for s, _, _, _ in top][::-1], color="#2980b9")
ax.set(xlabel="score (gap or |corr|)", title="15 features most associated with MICHD")
plt.tight_layout()
plt.show()

## 11. Candidates after the funnel

Intermediate list after the domain passes. We then adjust it (additions / drops) before writing the single file.


In [ ]:
FEATURES_BASE = [n for n in keep4 if n != "_STATE"]
print(f"{len(FEATURES_BASE)} features after funnel (before final additions/drops)")
for i in range(0, len(FEATURES_BASE), 6):
    print("  " + ", ".join(FEATURES_BASE[i:i + 6]))


## 12. Final feature list

I add 6 useful columns, drop `CVDSTRK3` (possible leakage), and write **a single** file: `selected_features.txt`.
The preprocessing reads only this file.


In [ ]:
ADD = ["GENHLTH", "POORHLTH", "PREDIAB1", "ALCDAY5", "CIMEMLOS", "BPMEDS"]
DROP = ["CVDSTRK3"]  # permanently excluded (too close to the MICHD label)
FEATURES = [n for n in FEATURES_BASE if n not in DROP]
for n in ADD:
    if n not in FEATURES:
        FEATURES.append(n)

path_feat = OUT / "selected_features.txt"
path_feat.write_text("\n".join(FEATURES) + "\n")
print(f"{len(FEATURES)} features → {path_feat}")
print("added:", ADD)
print("removed:", DROP)



## 13. CSV export (drop / keep)

I rebuild the lists **from this notebook** (passes 1→4 + adjustments) and write the CSVs.


In [ ]:
all_cols = list(feature_names)
kept_final = set(FEATURES)

# --- stage 1 ---
rows_drop = []
for n in all_cols:
    if n in NOT_RISK:
        rows_drop.append({
            "column": n,
            "stage": "1_not_a_risk_factor",
            "reason": "admin_phone_weights_screening_quality",
            "kept_instead": "",
            "note": "not a risk factor for MICHD",
        })

# --- stage 2 ---
for n in drop2:
    if n in kept_final:
        continue
    rows_drop.append({
        "column": n,
        "stage": "2_missing_ge_90pct",
        "reason": "empty_plus_DK_refused_ge_90",
        "kept_instead": "",
        "note": "88 is not a missing value",
    })

# --- CVDSTRK3 temporary drop ---
for n in DROP:
    rows_drop.append({
        "column": n,
        "stage": "3_label_leakage",
        "reason": "possible_leakage_close_to_MICHD",
        "kept_instead": "",
        "note": "temporary drop (stroke too close to CHD)",
    })

# --- stage 3: one version per concept ---
seen = {r["column"] for r in rows_drop}
for g in groups:
    names_g = [cols[i] for i in g]
    chosen = pick(g)
    for n in names_g:
        if n == chosen or n in seen or n in kept_final:
            continue
        rows_drop.append({
            "column": n,
            "stage": "3_correlation_ge_0.75",
            "reason": "same_concept",
            "kept_instead": chosen,
            "note": f"correlated >= {ASSOC_THRESHOLD} with {chosen}",
        })
        seen.add(n)

# --- stage 4 domain pass (on keep3, excluding v2) ---
for n in drop4:
    if n in seen or n in kept_final:
        continue
    rows_drop.append({
        "column": n,
        "stage": "4_domain_pass",
        "reason": "remaining_topic_or_skip_or_STATE",
        "kept_instead": "",
        "note": "DROP_METIER; some reopened in v2",
    })
    seen.add(n)

# everything else that is not kept
for n in all_cols:
    if n in kept_final or n in seen:
        continue
    rows_drop.append({
        "column": n,
        "stage": "4_domain_pass",
        "reason": "not_kept_after_funnel",
        "kept_instead": "",
        "note": "",
    })
    seen.add(n)

path_drop = OUT / "dropped_columns.csv"
with path_drop.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["column", "stage", "reason", "kept_instead", "note"])
    w.writeheader()
    for r in sorted(rows_drop, key=lambda x: (x["stage"], x["column"])):
        w.writerow(r)

print(f"dropped: {len(rows_drop)} → {path_drop}")
assert len(rows_drop) + len(FEATURES) == len(all_cols), (
    len(rows_drop), len(FEATURES), len(all_cols)
)


In [ ]:
from preprocess import (
    BINARY_YESNO, ORDINAL, NOMINAL, COUNT, CONTINUOUS, LOG1P,
)

FAMILY = {
    "_PACAT1": "CDC_risk", "_DRDXAR1": "CDC_risk", "_ASTHMS1": "CDC_risk",
    "_AGE80": "CDC_risk", "_SMOKER3": "CDC_risk", "_BMI5": "CDC_risk",
    "_CHOLCHK": "CDC_risk", "_RFBING5": "CDC_risk", "_FRUTSUM": "CDC_risk",
    "_VEGESUM": "CDC_risk", "_RFHLTH": "CDC_risk", "_RFHYPE5": "CDC_risk",
    "_RFCHOL": "CDC_risk", "_RFDRHV5": "CDC_risk", "_EDUCAG": "CDC_risk",
    "_INCOMG": "CDC_risk", "_RACE": "CDC_risk",
    "DIABETE3": "comorb_symptoms", "PREDIAB1": "comorb_symptoms",
    "CHCKIDNY": "comorb_symptoms", "CHCCOPD1": "comorb_symptoms", "ADDEPEV2": "comorb_symptoms",
    "CHCSCNCR": "comorb_symptoms", "CHCOCNCR": "comorb_symptoms", "GENHLTH": "comorb_symptoms",
    "POORHLTH": "comorb_symptoms", "PHYSHLTH": "comorb_symptoms", "MENTHLTH": "comorb_symptoms",
    "CIMEMLOS": "comorb_symptoms", "BPMEDS": "comorb_symptoms",
    "HLTHPLN1": "care_access", "PERSDOC2": "care_access", "MEDCOST": "care_access", "CHECKUP1": "care_access",
    "SEX": "demo_social", "MARITAL": "demo_social", "RENTHOM1": "demo_social", "VETERAN3": "demo_social",
    "EMPLOY1": "demo_social", "CHILDREN": "demo_social", "INTERNET": "demo_social",
    "SCNTMNY1": "demo_social", "SCNTMEL1": "demo_social", "SXORIENT": "demo_social", "TRNSGNDR": "demo_social",
    "QLACTLM2": "disability", "USEEQUIP": "disability", "BLIND": "disability", "DECIDE": "disability",
    "DIFFWALK": "disability", "DIFFDRES": "disability", "DIFFALON": "disability",
    "ALCDAY5": "lifestyle", "USENOW3": "lifestyle",
}

def typ_enc(n):
    if n in BINARY_YESNO:
        return "binary", "0_1"
    if n == "SEX":
        return "binary", "SEX_female"
    if n in NOMINAL:
        return "nominal", "one_hot"
    if n in ORDINAL:
        return "ordinal", "numeric"
    if n in COUNT:
        return "count", "numeric_88_to_0"
    if n in CONTINUOUS:
        if n in LOG1P:
            return "continuous", "log1p_then_scale"
        if n == "ALCDAY5":
            return "continuous", "recode_days_per_month_then_scale"
        return "continuous", "scale"
    return "other", "numeric"

path_cols = OUT / "kept_columns.csv"
with path_cols.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["column", "family", "type", "encoding", "note"])
    w.writeheader()
    for n in FEATURES:
        typ, enc = typ_enc(n)
        w.writerow({
            "column": n,
            "family": FAMILY.get(n, ""),
            "type": typ,
            "encoding": enc,
            "note": "added" if n in ADD else "",
        })

print(f"kept: {len(FEATURES)} → {path_cols}")
for i in range(0, len(FEATURES), 6):
    print("  " + ", ".join(FEATURES[i:i + 6]))


## 14. Preprocessing

I turn the features in `selected_features.txt` into a numeric matrix (`preprocess.py`):
recode, **`__isna` flags** (8 useful skip columns), impute (train), one-hot, scale (train), bias.

Isna only for: `BPMEDS`, `PREDIAB1`, `POORHLTH`, `CIMEMLOS`, `SCNTMNY1`, `SCNTMEL1`, `SXORIENT`, `TRNSGNDR`.


In [ ]:
import importlib
import preprocess
importlib.reload(preprocess)
from preprocess import build_train_test, MISSING_FLAG_COLS

tx, y01, tx_test, pp = build_train_test(
    x_train=X_TRAIN,
    y_train=Y_TRAIN,
    x_test=X_TEST,
    features_path=OUT / "selected_features.txt",
)
isna_cols = [n for n in pp.out_names_ if n.endswith("__isna")]
print("train tx", tx.shape, "  y01", y01.shape, f"  %MICHD {100 * y01.mean():.2f}%")
print("test  tx", tx_test.shape)
print("nan ?", np.isnan(tx).any(), np.isnan(tx_test).any())
print("bias ?", np.allclose(tx[:, 0], 1))
print(f"isna flags ({len(isna_cols)}):", isna_cols)
print("MISSING_FLAG_COLS :", sorted(MISSING_FLAG_COLS))
print("output columns:", len(pp.out_names_))
print("examples:", pp.out_names_[:8], "...")

assert "CVDSTRK3" not in pp.out_names_, "CVDSTRK3 still present — rerun §12 (DROP)"
assert "CVDSTRK3" not in (OUT / "selected_features.txt").read_text().splitlines()
print("OK: CVDSTRK3 absent from features and tx")



## 15. Models — F1 helpers + split

NumPy only. We maximize the **F1 of the MICHD=1 class**.
Each model produces a score (`tx @ w` or a probability); the **threshold** is a critical hyperparameter (imbalance ~9 %).


In [ ]:
import importlib
import implementations

importlib.reload(implementations)
from implementations import (
    least_squares,
    ridge_regression,
    logistic_regression,
    reg_logistic_regression,
    sigmoid,
)

SEED = 42
VAL_RATIO = 0.20
# None = everything. E.g. 80000 for a faster grid while debugging.
N_MAX = None

rng = np.random.default_rng(SEED)


def f1_bin(y_true, scores, t):
    """F1 / precision / recall for the positive class (y in {0,1})."""
    pred = (scores >= t).astype(np.float64)
    tp = np.sum((pred == 1) & (y_true == 1))
    fp = np.sum((pred == 1) & (y_true == 0))
    fn = np.sum((pred == 0) & (y_true == 1))
    prec = tp / (tp + fp + 1e-12)
    rec = tp / (tp + fn + 1e-12)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    return float(f1), float(prec), float(rec)


def best_threshold(y_true, scores, ts=None):
    if ts is None:
        # denser near low probs (rare class)
        ts = np.unique(np.concatenate([
            np.linspace(0.02, 0.40, 39),
            np.linspace(0.40, 0.80, 9),
        ]))
    best = (-1.0, None, 0.0, 0.0)
    for t in ts:
        f1, prec, rec = f1_bin(y_true, scores, t)
        if f1 > best[0]:
            best = (f1, float(t), prec, rec)
    return best  # f1, t, prec, rec


def stratified_split(y, val_ratio=VAL_RATIO, seed=SEED, n_max=N_MAX):
    y = np.asarray(y).ravel()
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    r = np.random.default_rng(seed)
    r.shuffle(idx0)
    r.shuffle(idx1)
    if n_max is not None:
        # keep the same positive rate
        n1 = max(1, int(round(n_max * y.mean())))
        n0 = n_max - n1
        idx0, idx1 = idx0[:n0], idx1[:n1]
    n0_va = int(round(len(idx0) * val_ratio))
    n1_va = int(round(len(idx1) * val_ratio))
    va = np.concatenate([idx0[:n0_va], idx1[:n1_va]])
    tr = np.concatenate([idx0[n0_va:], idx1[n1_va:]])
    r.shuffle(tr)
    r.shuffle(va)
    return tr, va


def linear_scores(tx, w):
    return tx @ w


def logistic_probs(tx, w):
    return sigmoid(tx @ w)


def logistic_weighted(y, tx, initial_w, max_iters, gamma, pos_weight=None):
    """Logistic GD with a weight on class 1 (imbalance). Returned loss = unweighted NLL."""
    y = np.asarray(y).ravel()
    w = np.asarray(initial_w, dtype=float).ravel().copy()
    n1 = float(y.sum())
    n0 = float(len(y) - n1)
    pw = (n0 / max(n1, 1.0)) if pos_weight is None else float(pos_weight)
    sw = np.where(y == 1.0, pw, 1.0)
    sw = sw / sw.mean()  # keeps gamma comparable
    for _ in range(max_iters):
        err = sw * (sigmoid(tx @ w) - y)
        w = w - gamma * (tx.T @ err) / sw.sum()
    z = tx @ w
    loss = float(np.mean(np.logaddexp(0.0, z) - y * z))
    return w, loss


tr, va = stratified_split(y01)
print(f"split train {len(tr)}  val {len(va)}  %MICHD tr={100*y01[tr].mean():.2f}%  va={100*y01[va].mean():.2f}%")
print(f"N_MAX={N_MAX}  (set an int to speed up)")


## 16. Grid search — least squares & ridge

Closed-form → fast. We grid over `λ` (ridge) and pick the **threshold** that maximizes F1 on val.


In [ ]:
results = []  # rows: method, params, f1, prec, rec, threshold, extra

# --- least squares ---
try:
    w_ls, mse_ls = least_squares(y01[tr], tx[tr])
    sc = linear_scores(tx[va], w_ls)
    f1, t, prec, rec = best_threshold(y01[va], sc)
    results.append({
        "method": "least_squares", "params": "—",
        "f1": f1, "prec": prec, "rec": rec, "threshold": t, "extra": f"mse={mse_ls:.5f}",
    })
    print(f"least_squares  threshold={t:.3f}  F1={f1:.4f}  prec={prec:.3f}  rec={rec:.3f}  mse={mse_ls:.5f}")
except np.linalg.LinAlgError as e:
    print("least_squares FAILED (X'X singular):", e)
    w_ls, sc = None, None

# --- ridge grid ---
LAMBDAS = [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0, 3.0]
print(f"\n{'lambda':>10}  {'thr':>6}  {'F1':>6}  {'prec':>6}  {'rec':>6}")
best_ridge = None
ridge_cache = {}  # lam -> (w, scores_va)

for lam in LAMBDAS:
    w, mse = ridge_regression(y01[tr], tx[tr], lam)
    sc = linear_scores(tx[va], w)
    ridge_cache[lam] = (w, sc)
    f1, t, prec, rec = best_threshold(y01[va], sc)
    print(f"{lam:10.4g}  {t:6.3f}  {f1:6.4f}  {prec:6.3f}  {rec:6.3f}")
    row = {
        "method": "ridge", "params": f"λ={lam:g}",
        "f1": f1, "prec": prec, "rec": rec, "threshold": t, "extra": f"mse={mse:.5f}",
    }
    results.append(row)
    if best_ridge is None or f1 > best_ridge["f1"]:
        best_ridge = {**row, "w": w, "lam": lam, "scores": sc}

print(f"\n→ best ridge: {best_ridge['params']}  threshold={best_ridge['threshold']:.3f}  F1={best_ridge['f1']:.4f}")


## 17. Grid search — logistic & reg_logistic

Scores = probabilities `sigmoid(tx @ w)`. Grid over `gamma`, `λ`, `max_iters`.
A bit slower (GD): start with the grid below, then increase `iters` on the best one.


In [ ]:
def init_w_logistic(y_tr, d):
    w0 = np.zeros(d)
    p = float(np.clip(y_tr.mean(), 1e-6, 1 - 1e-6))
    w0[0] = np.log(p / (1.0 - p))  # bias = log-odds
    return w0


GAMMAS = [0.1, 0.5, 1.0]
ITERS_GRID = [60, 120]
LAMBDAS_LOG = [0.0, 1e-5, 1e-4, 1e-3, 1e-2]  # 0.0 ≈ unregularized logistic, also via reg_*

print("=== logistic_regression (unregularized) ===")
print(f"{'gamma':>7}  {'iters':>5}  {'thr':>6}  {'F1':>6}  {'prec':>6}  {'rec':>6}  nll")
best_log = None
d = tx.shape[1]

for gamma in GAMMAS:
    for iters in ITERS_GRID:
        w0 = init_w_logistic(y01[tr], d)
        w, nll = logistic_regression(y01[tr], tx[tr], w0, max_iters=iters, gamma=gamma)
        probs = logistic_probs(tx[va], w)
        f1, t, prec, rec = best_threshold(y01[va], probs)
        print(f"{gamma:7.3f}  {iters:5d}  {t:6.3f}  {f1:6.4f}  {prec:6.3f}  {rec:6.3f}  {nll:.4f}")
        row = {
            "method": "logistic", "params": f"γ={gamma:g}, iters={iters}",
            "f1": f1, "prec": prec, "rec": rec, "threshold": t, "extra": f"nll={nll:.4f}",
        }
        results.append(row)
        if best_log is None or f1 > best_log["f1"]:
            best_log = {**row, "w": w, "probs": probs, "gamma": gamma, "iters": iters}

print(f"\n→ best logistic: {best_log['params']}  threshold={best_log['threshold']:.3f}  F1={best_log['f1']:.4f}")

print("\n=== reg_logistic_regression ===")
print(f"{'lambda':>10}  {'gamma':>6}  {'iters':>5}  {'thr':>6}  {'F1':>6}  {'prec':>6}  {'rec':>6}")
best_reg = None
# slightly smaller grid to save time
for lam in LAMBDAS_LOG:
    for gamma in [0.5, 1.0]:
        for iters in [120]:
            w0 = init_w_logistic(y01[tr], d)
            w, nll = reg_logistic_regression(
                y01[tr], tx[tr], lam, w0, max_iters=iters, gamma=gamma
            )
            probs = logistic_probs(tx[va], w)
            f1, t, prec, rec = best_threshold(y01[va], probs)
            print(f"{lam:10.4g}  {gamma:6.3f}  {iters:5d}  {t:6.3f}  {f1:6.4f}  {prec:6.3f}  {rec:6.3f}")
            row = {
                "method": "reg_logistic", "params": f"λ={lam:g}, γ={gamma:g}, iters={iters}",
                "f1": f1, "prec": prec, "rec": rec, "threshold": t, "extra": f"nll={nll:.4f}",
            }
            results.append(row)
            if best_reg is None or f1 > best_reg["f1"]:
                best_reg = {**row, "w": w, "probs": probs, "lam": lam, "gamma": gamma, "iters": iters}

print(f"\n→ best reg_logistic: {best_reg['params']}  threshold={best_reg['threshold']:.3f}  F1={best_reg['f1']:.4f}")


## 18. Weighted logistic (rare class)

Same idea as `class_weight='balanced'`: the gradient gives more weight to MICHD=1.
Local function (not in the graded `implementations.py`).


In [ ]:
print("=== logistic_weighted (pos_weight = n0/n1) ===")
print(f"{'gamma':>7}  {'iters':>5}  {'thr':>6}  {'F1':>6}  {'prec':>6}  {'rec':>6}")
best_wlog = None
for gamma in [0.1, 0.5, 1.0]:
    for iters in [80, 150]:
        w0 = init_w_logistic(y01[tr], d)
        w, nll = logistic_weighted(y01[tr], tx[tr], w0, max_iters=iters, gamma=gamma)
        probs = logistic_probs(tx[va], w)
        f1, t, prec, rec = best_threshold(y01[va], probs)
        print(f"{gamma:7.3f}  {iters:5d}  {t:6.3f}  {f1:6.4f}  {prec:6.3f}  {rec:6.3f}")
        row = {
            "method": "logistic_weighted", "params": f"γ={gamma:g}, iters={iters}, pw=balanced",
            "f1": f1, "prec": prec, "rec": rec, "threshold": t, "extra": f"nll={nll:.4f}",
        }
        results.append(row)
        if best_wlog is None or f1 > best_wlog["f1"]:
            best_wlog = {**row, "w": w, "probs": probs}

print(f"\n→ best weighted: {best_wlog['params']}  threshold={best_wlog['threshold']:.3f}  F1={best_wlog['f1']:.4f}")


## 19. Score ensembles + final ranking

Average (possibly weighted) of the best models' scores, then an F1 threshold on val.


In [ ]:
def to_unit_scores(scores):
    """Maps a linear score into (0,1) via rank → comparable to probabilities."""
    # rank percentile
    order = scores.argsort().argsort().astype(np.float64)
    return (order + 1.0) / (len(scores) + 1.0)


# val scores of the best models
cands = []
if best_ridge is not None:
    cands.append(("ridge", to_unit_scores(best_ridge["scores"])))
if best_log is not None:
    cands.append(("logistic", best_log["probs"]))
if best_reg is not None:
    cands.append(("reg_logistic", best_reg["probs"]))
if best_wlog is not None:
    cands.append(("weighted", best_wlog["probs"]))

print("ensembles tested:")
ens_best = None
# pairs + average of all
from itertools import combinations
names = [c[0] for c in cands]
arrs = [c[1] for c in cands]

combos = []
for k in range(2, len(cands) + 1):
    for idxs in combinations(range(len(cands)), k):
        combos.append(idxs)

for idxs in combos:
    label = "+".join(names[i] for i in idxs)
    sc = np.mean([arrs[i] for i in idxs], axis=0)
    f1, t, prec, rec = best_threshold(y01[va], sc)
    print(f"  {label:<40}  threshold={t:.3f}  F1={f1:.4f}")
    row = {
        "method": "ensemble", "params": label,
        "f1": f1, "prec": prec, "rec": rec, "threshold": t, "extra": "",
    }
    results.append(row)
    if ens_best is None or f1 > ens_best["f1"]:
        ens_best = row

print(f"\n→ best ensemble: {ens_best['params']}  F1={ens_best['f1']:.4f}")


In [ ]:
# Overall ranking
rows = sorted(results, key=lambda r: -r["f1"])
print(f"{'method':<20}  {'params':<42}  {'F1':>6}  {'prec':>6}  {'rec':>6}  {'thr':>6}")
print("-" * 100)
for r in rows[:25]:
    print(f"{r['method']:<20}  {r['params']:<42}  {r['f1']:6.4f}  {r['prec']:6.3f}  {r['rec']:6.3f}  {r['threshold']:6.3f}")

best_all = rows[0]
print(f"\n★ BEST val F1 = {best_all['f1']:.4f}   [{best_all['method']}]  {best_all['params']}  threshold={best_all['threshold']:.3f}")

# save for later
path_res = OUT / "model_grid_results.csv"
with path_res.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["method", "params", "f1", "prec", "rec", "threshold", "extra"])
    w.writeheader()
    for r in rows:
        w.writerow({k: r[k] for k in w.fieldnames})
print("saved →", path_res)


## 20. Refine the winner (more iters / neighbouring λ)

We refine around the best non-ensemble model. Then we can refit on the **whole** train set for the submission.


In [ ]:
# Ridge refinement around the best λ
if best_ridge is not None:
    lam0 = best_ridge["lam"]
    refine_lams = sorted(set([
        lam0,
        lam0 / 3 if lam0 > 0 else 1e-6,
        lam0 * 3 if lam0 > 0 else 1e-5,
        max(lam0 / 10, 0.0),
        lam0 * 10,
    ]))
    print("refine ridge λ :", refine_lams)
    best_ref = best_ridge
    for lam in refine_lams:
        w, mse = ridge_regression(y01[tr], tx[tr], lam)
        sc = linear_scores(tx[va], w)
        f1, t, prec, rec = best_threshold(y01[va], sc, ts=np.linspace(0.01, 0.5, 99))
        print(f"  λ={lam:<10.4g}  threshold={t:.3f}  F1={f1:.4f}")
        if f1 > best_ref["f1"]:
            best_ref = {"method": "ridge", "params": f"λ={lam:g} (refined)", "f1": f1,
                        "prec": prec, "rec": rec, "threshold": t, "w": w, "lam": lam, "scores": sc}
    print(f"→ refined ridge F1={best_ref['f1']:.4f}")

# Logistic refinement: more iters on the best (γ, λ)
if best_reg is not None:
    print("\nrefine reg_logistic (iters↑)")
    w0 = init_w_logistic(y01[tr], d)
    w, nll = reg_logistic_regression(
        y01[tr], tx[tr], best_reg["lam"], w0,
        max_iters=max(250, best_reg["iters"]), gamma=best_reg["gamma"],
    )
    probs = logistic_probs(tx[va], w)
    f1, t, prec, rec = best_threshold(y01[va], probs, ts=np.linspace(0.02, 0.45, 87))
    print(f"  F1={f1:.4f}  threshold={t:.3f}  nll={nll:.4f}  (before {best_reg['f1']:.4f})")
    if f1 > best_reg["f1"]:
        best_reg.update(f1=f1, threshold=t, prec=prec, rec=rec, w=w, probs=probs, extra=f"nll={nll:.4f}")
        print("  improved → keeping it")


## 21. AICrowd submission — best ensemble

According to `model_grid_results.csv`, the best is **`logistic + reg_logistic`** (threshold ~0.20).

Pipeline:
1. keep the threshold chosen on **val**
2. **refit** both models on the **whole** train set (`tx`, `y01`)
3. average the test probabilities → threshold → labels `{-1, +1}`
4. `Id,Prediction` CSV → upload to AICrowd


In [ ]:
from datetime import datetime
from helpers import create_csv_submission

# --- hyperparams of the best ensemble (adjust if your CSV says otherwise) ---
ENS_MODELS = ("logistic", "reg_logistic")  # same as the top of the grid
GAMMA_SUB = 1.0
ITERS_SUB = 120
LAM_REG = 1e-5
# threshold = that of the best ensemble on val (recomputed below if best_* is available)
THR_SUB = None

# 1) threshold: from the grid if available, otherwise recompute it on the current val
if "ens_best" in dir() and ens_best is not None and ens_best.get("params") == "logistic+reg_logistic":
    THR_SUB = ens_best["threshold"]
elif THR_SUB is None:
    # recreate val scores from current best_log / best_reg
    sc_va = 0.5 * (best_log["probs"] + best_reg["probs"])
    f1_chk, THR_SUB, prec_chk, rec_chk = best_threshold(y01[va], sc_va)
    print(f"threshold recomputed on val: {THR_SUB:.3f}  F1={f1_chk:.4f}")

print(f"submission ensemble {ENS_MODELS}  γ={GAMMA_SUB}  iters={ITERS_SUB}  λ_reg={LAM_REG}  threshold={THR_SUB:.3f}")

# 2) refit on the WHOLE train set
d = tx.shape[1]
w0 = init_w_logistic(y01, d)

w_log, nll_log = logistic_regression(y01, tx, w0.copy(), max_iters=ITERS_SUB, gamma=GAMMA_SUB)
w_reg, nll_reg = reg_logistic_regression(y01, tx, LAM_REG, w0.copy(), max_iters=ITERS_SUB, gamma=GAMMA_SUB)
print(f"refit full train  nll_log={nll_log:.4f}  nll_reg={nll_reg:.4f}")

# 3) test scores + average + threshold
prob_te = 0.5 * (logistic_probs(tx_test, w_log) + logistic_probs(tx_test, w_reg))
y_pred = np.where(prob_te >= THR_SUB, 1, -1).astype(int)

# test Ids (1st column of x_test.csv)
with X_TEST.open(newline="") as f:
    r = csv.reader(f)
    header = next(r)
    id_col = header.index("Id")
    ids_test = np.array([int(row[id_col]) for row in r])

assert len(ids_test) == len(y_pred), (len(ids_test), len(y_pred))
print(f"test n={len(y_pred)}  %pred MICHD={100*(y_pred==1).mean():.2f}%  (train had {100*y01.mean():.2f}%)")

# 4) write the CSV
SUBS = PROJ / "submissions"
SUBS.mkdir(exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
fname = SUBS / f"{stamp}_ens_log+reg_thr{THR_SUB:.3f}.csv"
create_csv_submission(ids_test, y_pred, str(fname))
print("→ file ready:", fname.resolve())
print("\nUpload: AICrowd → MICHD challenge / Project 1 → Submit → pick this CSV.")


## 22. Error analysis (where the model fails)

We start from the best ensemble **logistic + reg_logistic** on **val**, with its F1 threshold.
Goal: profile the **FN** (missed MICHD) vs **TP**, and the **FP** vs **TN**, to guide feature engineering.


In [ ]:
# --- val scores of the best ensemble (recompute if needed) ---
if "best_log" not in dir() or best_log is None or "best_reg" not in dir() or best_reg is None:
    raise RuntimeError("Rerun §§15–19 first (best_log / best_reg required).")

probs_va = 0.5 * (best_log["probs"] + best_reg["probs"])
if "ens_best" in dir() and ens_best is not None:
    thr_err = ens_best["threshold"]
else:
    _, thr_err, _, _ = best_threshold(y01[va], probs_va)

y_va = y01[va]
pred_va = (probs_va >= thr_err).astype(np.float64)

tp = (pred_va == 1) & (y_va == 1)
fp = (pred_va == 1) & (y_va == 0)
fn = (pred_va == 0) & (y_va == 1)
tn = (pred_va == 0) & (y_va == 0)

print(f"threshold={thr_err:.3f}")
print(f"TP={tp.sum():,}  FP={fp.sum():,}  FN={fn.sum():,}  TN={tn.sum():,}")
print(f"among true MICHD (val): recall = TP/(TP+FN) = {tp.sum()/(tp.sum()+fn.sum()):.3f}")
print(f"among predicted positives: precision = TP/(TP+FP) = {tp.sum()/(tp.sum()+fp.sum()+1e-12):.3f}")
print(f"\n→ we still miss {fn.sum():,} positives (FN) and raise {fp.sum():,} false alarms (FP)")


In [ ]:
# Mean feature profile (columns already standardized, bias excluded)
names = np.array(pp.out_names_)
Xva = tx[va]
# ignore bias col 0
feat_idx = np.arange(1, Xva.shape[1])
feat_names = names[feat_idx]

def group_mean(mask):
    if mask.sum() == 0:
        return np.full(len(feat_idx), np.nan)
    return Xva[mask][:, feat_idx].mean(axis=0)

means = {
    "TP": group_mean(tp),
    "FN": group_mean(fn),
    "FP": group_mean(fp),
    "TN": group_mean(tn),
    "pos": group_mean(y_va == 1),
    "neg": group_mean(y_va == 0),
}

# Most informative gaps
delta_fn_tp = means["FN"] - means["TP"]   # where the missed cases differ from the correctly detected ones
delta_fp_tn = means["FP"] - means["TN"]   # where the false alarms differ from the true negatives

def top_abs(delta, k=15):
    order = np.argsort(-np.abs(delta))
    rows = []
    for i in order[:k]:
        rows.append((feat_names[i], float(delta[i]), float(means["FN"][i]), float(means["TP"][i]),
                     float(means["FP"][i]), float(means["TN"][i])))
    return rows

print("=== FN − TP (features where missed MICHD ≠ correctly detected MICHD) ===")
print(f"{'feature':<28}  {'ΔFN-TP':>8}  {'FN':>7}  {'TP':>7}")
for name, d, fnm, tpm, _, _ in top_abs(delta_fn_tp, 15):
    print(f"{name:<28}  {d:8.3f}  {fnm:7.3f}  {tpm:7.3f}")

print("\n=== FP − TN (features where false alarms ≠ truly healthy) ===")
print(f"{'feature':<28}  {'ΔFP-TN':>8}  {'FP':>7}  {'TN':>7}")
# reuse printer with FP/TN cols
order = np.argsort(-np.abs(delta_fp_tn))
for i in order[:15]:
    print(f"{feat_names[i]:<28}  {delta_fp_tn[i]:8.3f}  {means['FP'][i]:7.3f}  {means['TN'][i]:7.3f}")


In [ ]:
# Small viz: top FN vs TP gaps
k_plot = 12
order = np.argsort(-np.abs(delta_fn_tp))[:k_plot]
labels = feat_names[order]
vals = delta_fn_tp[order]
colors = ["#c0392b" if v > 0 else "#2980b9" for v in vals]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(labels[::-1], vals[::-1], color=colors[::-1])
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("mean (standardized) FN − TP")
ax.set_title("Where false negatives differ from true positives")
plt.tight_layout()
plt.show()

# Scores: are the FN just below the threshold?
print(f"proba val  TP: mean={probs_va[tp].mean():.3f}  median={np.median(probs_va[tp]):.3f}")
print(f"proba val  FN: mean={probs_va[fn].mean():.3f}  median={np.median(probs_va[fn]):.3f}  (threshold={thr_err:.3f})")
print(f"proba val  FP: mean={probs_va[fp].mean():.3f}  median={np.median(probs_va[fp]):.3f}")
print(f"\nFN just below the threshold (thr-0.05, thr): {np.sum(fn & (probs_va >= thr_err - 0.05)):,} / {fn.sum():,}")


## 23. Error-driven feature engineering

We add targeted **domain interactions** (products of columns already in `tx`),
often related to the FN/TP gaps: age × hypertension / chol / smoker / diabetes / self-rated health, etc.

Then we **re-standardize** the new columns (train-only stats) and retest
ridge + logistic + ensemble on the **same** split.


In [ ]:
def col(name):
    """Index of an output column (after preprocess)."""
    return pp.out_names_.index(name)


def safe_col(*candidates):
    for n in candidates:
        if n in pp.out_names_:
            return n
    return None


# interaction pairs (names after preprocess)
INTER_PAIRS = []
candidates = [
    ("_AGE80", "_RFHYPE5"),
    ("_AGE80", "_RFCHOL"),
    ("_AGE80", "_SMOKER3"),
    ("_AGE80", "GENHLTH"),
    ("_BMI5", "_RFHYPE5"),
    ("_BMI5", "_RFCHOL"),
    ("_RFHYPE5", "_RFCHOL"),
    ("_SMOKER3", "_RFCHOL"),
    ("_RFHLTH", "DIFFWALK"),
    ("GENHLTH", "_RFHYPE5"),
    ("_AGE80", "SEX_female") if "SEX_female" in pp.out_names_ else ("_AGE80", "SEX"),
]
# diabetes one-hot if present
for diab in ["DIABETE3=1", "DIABETE3=2", "DIABETE3=3", "DIABETE3=4"]:
    if diab in pp.out_names_:
        candidates.append(("_AGE80", diab))
        candidates.append(("_RFHYPE5", diab))
        break

for a, b in candidates:
    if a in pp.out_names_ and b in pp.out_names_:
        INTER_PAIRS.append((a, b))

# never CVDSTRK3 in the interactions
INTER_PAIRS = [(a, b) for a, b in INTER_PAIRS if "CVDSTRK3" not in (a, b)]

print(f"{len(INTER_PAIRS)} interactions:")
for a, b in INTER_PAIRS:
    print(f"  {a} * {b}")


def add_interactions(tx_mat, pairs, fit_mean=None, fit_std=None):
    """Adds products; standardizes the new cols (fit on train if fit_*=None)."""
    extras = []
    names_extra = []
    for a, b in pairs:
        extras.append(tx_mat[:, col(a)] * tx_mat[:, col(b)])
        names_extra.append(f"{a}*{b}")
    E = np.column_stack(extras)
    if fit_mean is None:
        fit_mean = E.mean(axis=0)
        fit_std = E.std(axis=0)
        fit_std[fit_std == 0] = 1.0
    E = (E - fit_mean) / fit_std
    return np.c_[tx_mat, E], names_extra, fit_mean, fit_std


tx2_tr, inter_names, inter_mean, inter_std = add_interactions(tx[tr], INTER_PAIRS)
tx2_va, _, _, _ = add_interactions(tx[va], INTER_PAIRS, inter_mean, inter_std)
# full matrices for optional later use
tx2, _, _, _ = add_interactions(tx, INTER_PAIRS, inter_mean, inter_std)
tx2_test, _, _, _ = add_interactions(tx_test, INTER_PAIRS, inter_mean, inter_std)

print(f"tx : {tx.shape[1]} → {tx2.shape[1]} columns (+{len(inter_names)})")



In [ ]:
# Quick retest on the same split
results_fe = []

def eval_linear(label, w, Xva, yva):
    sc = Xva @ w
    f1, t, prec, rec = best_threshold(yva, sc)
    results_fe.append({"method": label, "f1": f1, "thr": t, "prec": prec, "rec": rec})
    print(f"{label:<28}  threshold={t:.3f}  F1={f1:.4f}  prec={prec:.3f}  rec={rec:.3f}")
    return f1, t, sc


def eval_logistic(label, w, Xva, yva):
    pr = sigmoid(Xva @ w)
    f1, t, prec, rec = best_threshold(yva, pr)
    results_fe.append({"method": label, "f1": f1, "thr": t, "prec": prec, "rec": rec, "probs": pr, "w": w})
    print(f"{label:<28}  threshold={t:.3f}  F1={f1:.4f}  prec={prec:.3f}  rec={rec:.3f}")
    return f1, t, pr, w


print("=== without interactions (grid reference) ===")
print(f"  ensemble log+reg          F1≈{ens_best['f1']:.4f}  threshold={ens_best['threshold']:.3f}")

print("\n=== with interactions ===")
# ridge
for lam in [1e-3, 1e-2, 0.1]:
    w, _ = ridge_regression(y01[tr], tx2_tr, lam)
    eval_linear(f"ridge λ={lam:g} +inter", w, tx2_va, y01[va])

# logistic / reg
d2 = tx2_tr.shape[1]
w0 = init_w_logistic(y01[tr], d2)
w_log2, _ = logistic_regression(y01[tr], tx2_tr, w0.copy(), max_iters=120, gamma=1.0)
_, _, pr_log2, w_log2 = eval_logistic("logistic +inter", w_log2, tx2_va, y01[va])

w_reg2, _ = reg_logistic_regression(y01[tr], tx2_tr, 1e-5, w0.copy(), max_iters=120, gamma=1.0)
_, _, pr_reg2, w_reg2 = eval_logistic("reg_logistic +inter", w_reg2, tx2_va, y01[va])

# ensemble
pr_ens2 = 0.5 * (pr_log2 + pr_reg2)
f1e, te, prece, rece = best_threshold(y01[va], pr_ens2)
results_fe.append({"method": "ensemble log+reg +inter", "f1": f1e, "thr": te, "prec": prece, "rec": rece})
print(f"{'ensemble log+reg +inter':<28}  threshold={te:.3f}  F1={f1e:.4f}  prec={prece:.3f}  rec={rece:.3f}")

print(f"\nΔ F1 ensemble: {f1e - ens_best['f1']:+.4f}  (before {ens_best['f1']:.4f} → after {f1e:.4f})")


## 24. Which interactions really help?

We look at the `|w|` of the new columns in logistic+inter, and redo an
FN/TP error pass **after** FE to see whether the profile has moved.


In [ ]:
# Interaction importance (logistic weights)
w_inter = w_log2[-len(inter_names):]
order = np.argsort(-np.abs(w_inter))
print(f"{'interaction':<40}  {'w':>9}")
for i in order:
    print(f"{inter_names[i]:<40}  {w_inter[i]:9.4f}")

# New confusion + top Δ FN-TP on base features only
pred2 = (pr_ens2 >= te).astype(np.float64)
tp2 = (pred2 == 1) & (y_va == 1)
fn2 = (pred2 == 0) & (y_va == 1)
fp2 = (pred2 == 1) & (y_va == 0)
print(f"\nafter FE: TP={tp2.sum():,}  FP={fp2.sum():,}  FN={fn2.sum():,}  (before FN={fn.sum():,} FP={fp.sum():,})")
print(f"FN avoided: {int(fn.sum() - fn2.sum()):+d}    extra FP: {int(fp2.sum() - fp.sum()):+d}")

# Save the best one for a possible submission
best_fe = max(results_fe, key=lambda r: r["f1"])
print(f"\n★ best after FE: {best_fe['method']}  F1={best_fe['f1']:.4f}  threshold={best_fe['thr']:.3f}")
if best_fe["f1"] > ens_best["f1"]:
    print("→ the interactions help: you can regenerate a submission with tx2 / tx2_test (adapted §21).")
else:
    print("→ no clear gain: keep the model without interactions, or rework other features (drops / isna / raw).")



## 25. Top-15 interactions only (no noise, no CVDSTRK3)

We keep the **15 products** with the largest `|w|` in logistic+inter (§24),
rebuild `tx`, and retest the models on the same split.


In [ ]:
TOP_K = 15

# CVDSTRK3: excluded from the interactions; warn only if it is still in tx
if "CVDSTRK3" in pp.out_names_:
    print("⚠ CVDSTRK3 still in tx — continuing without using it in the interactions")

# ranking by |w| of the logistic trained with all interactions (§23)
w_inter_all = np.asarray(w_log2[-len(inter_names):]).ravel()
order = np.argsort(-np.abs(w_inter_all))

# candidates without CVDSTRK3
cands = [i for i in order if "CVDSTRK3" not in inter_names[i] and "CVDSTRK3" not in INTER_PAIRS[i]]
top_idx = cands[:TOP_K]
TOP_PAIRS = [INTER_PAIRS[i] for i in top_idx]
TOP_NAMES = [inter_names[i] for i in top_idx]

print(f"Top-{len(TOP_PAIRS)} interactions (|w| logistic), without CVDSTRK3:")
print(f"{'interaction':<40}  {'|w|':>8}  {'w':>9}")
for i in top_idx:
    print(f"{inter_names[i]:<40}  {abs(w_inter_all[i]):8.4f}  {w_inter_all[i]:9.4f}")

# rebuild the matrices with ONLY these interactions
tx15_tr, names15, mean15, std15 = add_interactions(tx[tr], TOP_PAIRS)
tx15_va, _, _, _ = add_interactions(tx[va], TOP_PAIRS, mean15, std15)
tx15, _, _, _ = add_interactions(tx, TOP_PAIRS, mean15, std15)
tx15_test, _, _, _ = add_interactions(tx_test, TOP_PAIRS, mean15, std15)

print(f"\ntx : {tx.shape[1]} → {tx15.shape[1]}  (+{len(names15)} inter)")
print("inter OK without CVDSTRK3:", all("CVDSTRK3" not in n for n in names15))



In [ ]:
results_top15 = []

def _track(method, f1, thr, prec, rec, **extra):
    row = {"method": method, "f1": f1, "thr": thr, "prec": prec, "rec": rec, **extra}
    results_top15.append(row)
    print(f"{method:<32}  threshold={thr:.3f}  F1={f1:.4f}  prec={prec:.3f}  rec={rec:.3f}")
    return row


print("=== top-15 interactions — model retest ===")
# ridge
best_ridge15 = None
for lam in [1e-4, 1e-3, 1e-2, 0.1]:
    w, _ = ridge_regression(y01[tr], tx15_tr, lam)
    f1, t, prec, rec = best_threshold(y01[va], tx15_va @ w)
    row = _track(f"ridge λ={lam:g} +top15", f1, t, prec, rec, w=w, lam=lam)
    if best_ridge15 is None or f1 > best_ridge15["f1"]:
        best_ridge15 = row

# logistic / reg / weighted
d15 = tx15_tr.shape[1]
w0 = init_w_logistic(y01[tr], d15)

w_log15, nll = logistic_regression(y01[tr], tx15_tr, w0.copy(), max_iters=120, gamma=1.0)
pr = sigmoid(tx15_va @ w_log15)
f1, t, prec, rec = best_threshold(y01[va], pr)
best_log15 = _track("logistic +top15", f1, t, prec, rec, w=w_log15, probs=pr, nll=nll)

w_reg15, nll = reg_logistic_regression(y01[tr], tx15_tr, 1e-5, w0.copy(), max_iters=120, gamma=1.0)
pr = sigmoid(tx15_va @ w_reg15)
f1, t, prec, rec = best_threshold(y01[va], pr)
best_reg15 = _track("reg_logistic +top15", f1, t, prec, rec, w=w_reg15, probs=pr, nll=nll)

w_w15, nll = logistic_weighted(y01[tr], tx15_tr, w0.copy(), max_iters=150, gamma=0.5)
pr = sigmoid(tx15_va @ w_w15)
f1, t, prec, rec = best_threshold(y01[va], pr)
best_w15 = _track("logistic_weighted +top15", f1, t, prec, rec, w=w_w15, probs=pr, nll=nll)

# ensembles
pr_ens = 0.5 * (best_log15["probs"] + best_reg15["probs"])
f1, t, prec, rec = best_threshold(y01[va], pr_ens)
_track("ensemble log+reg +top15", f1, t, prec, rec, probs=pr_ens)

pr_ens2 = 0.5 * (best_log15["probs"] + best_w15["probs"])
f1, t, prec, rec = best_threshold(y01[va], pr_ens2)
_track("ensemble log+weighted +top15", f1, t, prec, rec, probs=pr_ens2)

# ranking
rows = sorted(results_top15, key=lambda r: -r["f1"])
print("\n=== top-15 ranking ===")
print(f"{'method':<32}  {'F1':>6}  {'thr':>6}  {'prec':>6}  {'rec':>6}")
print("-" * 64)
for r in rows:
    print(f"{r['method']:<32}  {r['f1']:6.4f}  {r['thr']:6.3f}  {r['prec']:6.3f}  {r['rec']:6.3f}")

best15 = rows[0]
print(f"\n★ BEST top-15 : {best15['method']}  F1={best15['f1']:.4f}  threshold={best15['thr']:.3f}")
if "ens_best" in dir() and ens_best is not None:
    print(f"  vs ensemble without inter: {ens_best['f1']:.4f}  (Δ={best15['f1']-ens_best['f1']:+.4f})")
if "best_fe" in dir() and best_fe is not None:
    print(f"  vs all inter             : {best_fe['f1']:.4f}  (Δ={best15['f1']-best_fe['f1']:+.4f})")


## 26. Submission with top-15 (if its F1 is the best)

Refit the best top-15 model on the **whole** train set → AICrowd CSV.


In [ ]:
from datetime import datetime
from helpers import create_csv_submission

# automatic choice: best of the top-15 ranking
method = best15["method"]
thr15 = best15["thr"]
print("submission:", method, "threshold=", thr15)

w0_full = init_w_logistic(y01, tx15.shape[1])

if method.startswith("ridge"):
    w_sub, _ = ridge_regression(y01, tx15, best_ridge15["lam"])
    scores_te = tx15_test @ w_sub
elif "weighted" in method and "ensemble" not in method:
    w_sub, _ = logistic_weighted(y01, tx15, w0_full, max_iters=150, gamma=0.5)
    scores_te = sigmoid(tx15_test @ w_sub)
elif method.startswith("reg_logistic"):
    w_sub, _ = reg_logistic_regression(y01, tx15, 1e-5, w0_full, max_iters=120, gamma=1.0)
    scores_te = sigmoid(tx15_test @ w_sub)
elif method.startswith("logistic +"):
    w_sub, _ = logistic_regression(y01, tx15, w0_full, max_iters=120, gamma=1.0)
    scores_te = sigmoid(tx15_test @ w_sub)
elif "log+reg" in method:
    wl, _ = logistic_regression(y01, tx15, w0_full.copy(), max_iters=120, gamma=1.0)
    wr, _ = reg_logistic_regression(y01, tx15, 1e-5, w0_full.copy(), max_iters=120, gamma=1.0)
    scores_te = 0.5 * (sigmoid(tx15_test @ wl) + sigmoid(tx15_test @ wr))
elif "log+weighted" in method:
    wl, _ = logistic_regression(y01, tx15, w0_full.copy(), max_iters=120, gamma=1.0)
    ww, _ = logistic_weighted(y01, tx15, w0_full.copy(), max_iters=150, gamma=0.5)
    scores_te = 0.5 * (sigmoid(tx15_test @ wl) + sigmoid(tx15_test @ ww))
else:
    raise RuntimeError("unhandled method: " + method)

y_pred = np.where(scores_te >= thr15, 1, -1).astype(int)

with X_TEST.open(newline="") as f:
    r = csv.reader(f)
    header = next(r)
    id_col = header.index("Id")
    ids_test = np.array([int(row[id_col]) for row in r])

assert len(ids_test) == len(y_pred)
assert "CVDSTRK3" not in names15

SUBS = PROJ / "submissions"
SUBS.mkdir(exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
fname = SUBS / f"{stamp}_top15_{method.replace(' ', '_')}_f1{best15['f1']:.4f}.csv"
create_csv_submission(ids_test, y_pred, str(fname))
print("→", fname.resolve())
print(f"%pred +1 = {100*(y_pred==1).mean():.2f}%")
print("Upload to AICrowd if you want to compare with the previous one.")


## 27. Model on the 15 features most correlated with MICHD

Anti-noise idea: keep only the **bias + top-15 columns** of `tx` most
correlated (Pearson) with `y` **on train**, without `CVDSTRK3`.
Then retest ridge / logistic / ensemble on the same split.


In [ ]:
K_FEAT = 15

names = np.array(pp.out_names_)
# candidate columns = everything except bias; ignore CVDSTRK3 if present
cand = [j for j, n in enumerate(names) if n != "bias" and "CVDSTRK3" not in n]

# |corr| correlation on TRAIN only (no val leakage)
y_tr = y01[tr]
corrs = np.zeros(len(cand))
for k, j in enumerate(cand):
    xj = tx[tr, j]
    if xj.std() < 1e-12:
        corrs[k] = 0.0
    else:
        corrs[k] = np.corrcoef(xj, y_tr)[0, 1]

order = np.argsort(-np.abs(corrs))
top = order[:K_FEAT]
TOP_FEAT_IDX = [cand[i] for i in top]          # indices in tx
TOP_FEAT_NAMES = [names[j] for j in TOP_FEAT_IDX]
TOP_FEAT_CORR = [corrs[i] for i in top]

print(f"Top-{K_FEAT} features |corr| with MICHD (train), without CVDSTRK3:")
print(f"{'feature':<28}  {'corr':>8}")
for n, c in zip(TOP_FEAT_NAMES, TOP_FEAT_CORR):
    print(f"{n:<28}  {c:8.4f}")

# matrices: bias (col 0) + top-15
cols_keep = [0] + TOP_FEAT_IDX
txK_tr = tx[tr][:, cols_keep]
txK_va = tx[va][:, cols_keep]
txK = tx[:, cols_keep]
txK_test = tx_test[:, cols_keep]
print(f"\nreduced tx: {tx.shape[1]} → {txK.shape[1]} columns (bias + {K_FEAT})")


In [ ]:
results_k15 = []

def track(method, f1, thr, prec, rec, **extra):
    row = {"method": method, "f1": f1, "thr": thr, "prec": prec, "rec": rec, **extra}
    results_k15.append(row)
    print(f"{method:<30}  threshold={thr:.3f}  F1={f1:.4f}  P={prec:.3f}  R={rec:.3f}")
    return row

print("=== models on bias + top-15 corr(MICHD) ===")
best_ridge_k = None
for lam in [1e-4, 1e-3, 1e-2, 0.1, 1.0]:
    w, _ = ridge_regression(y01[tr], txK_tr, lam)
    f1, t, p, r = best_threshold(y01[va], txK_va @ w)
    row = track(f"ridge λ={lam:g}", f1, t, p, r, w=w, lam=lam)
    if best_ridge_k is None or f1 > best_ridge_k["f1"]:
        best_ridge_k = row

dK = txK_tr.shape[1]
w0 = init_w_logistic(y01[tr], dK)

w, nll = logistic_regression(y01[tr], txK_tr, w0.copy(), 120, 1.0)
pr = sigmoid(txK_va @ w)
f1, t, p, r = best_threshold(y01[va], pr)
best_log_k = track("logistic", f1, t, p, r, w=w, probs=pr)

w, nll = reg_logistic_regression(y01[tr], txK_tr, 1e-5, w0.copy(), 120, 1.0)
pr = sigmoid(txK_va @ w)
f1, t, p, r = best_threshold(y01[va], pr)
best_reg_k = track("reg_logistic", f1, t, p, r, w=w, probs=pr)

w, nll = logistic_weighted(y01[tr], txK_tr, w0.copy(), 150, 0.5)
pr = sigmoid(txK_va @ w)
f1, t, p, r = best_threshold(y01[va], pr)
best_w_k = track("logistic_weighted", f1, t, p, r, w=w, probs=pr)

pr = 0.5 * (best_log_k["probs"] + best_reg_k["probs"])
f1, t, p, r = best_threshold(y01[va], pr)
track("ensemble log+reg", f1, t, p, r, probs=pr)

rows = sorted(results_k15, key=lambda x: -x["f1"])
print("\n=== ranking ===")
for row in rows:
    print(f"{row['method']:<30}  F1={row['f1']:.4f}  thr={row['thr']:.3f}")

best_k15 = rows[0]
print(f"\n★ BEST top-15 features: {best_k15['method']}  F1={best_k15['f1']:.4f}")
if "ens_best" in dir() and ens_best is not None:
    print(f"  vs full model (ens without inter): {ens_best['f1']:.4f}  Δ={best_k15['f1']-ens_best['f1']:+.4f}")
print("If Δ is strongly negative → too much signal lost; keep the full model.")


## 28. Improvement sweep (weighted / poly / silent FN / CV)

Result already computed outside the notebook → `improve_sweep.txt`.

**Hold-out**: slight gain with `poly` (squares of age/BMI/GENHLTH/smoker) + logistic ≈ **0.426**.
**3-fold CV**: everything is within **0.413–0.415**; best = `poly | ens log+reg` ≈ **0.415**.
Weighted / silent FN / poly+silent: **no better** than that (noise).

Conclusion: we are on a **plateau**; keep **full or poly + logistic/ensemble**, threshold ~0.20.


In [ ]:
# show the saved sweep
path = OUT / "improve_sweep.txt"
print(path.read_text() if path.exists() else "file missing — rerun the sweep script")
